In [ ]:
import numpy as np
import cv2
import json
import os
import random
from scipy.interpolate import splprep, splev
from pathlib import Path

# Configuration
IMG_SIZE = (512, 512)           # Height, Width
NUM_IMAGES_PER_TIER = 100
OUTPUT_DIR = "synthetic_fault_lines_realistic"

TIERS = ["tier1_clean", "tier2_noise_gaps", "tier3_curved_cross", 
         "tier4_dense_overlap", "tier5_hard"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
for tier in TIERS:
    os.makedirs(os.path.join(OUTPUT_DIR, tier), exist_ok=True)

def random_spline_points(n_points=8, curvature=0.2):
    """Minimal gentle curves only — almost straight when used"""
    x = np.linspace(40, IMG_SIZE[1]-40, n_points) + np.random.uniform(-10, 10, n_points)
    y = np.linspace(80, IMG_SIZE[0]-80, n_points) + np.random.normal(0, 5 + 12*curvature, n_points)
    
    pts = np.column_stack((x, y))
    
    angle = np.random.uniform(-np.pi/8, np.pi/8)
    rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    pts = pts @ rot.T
    
    pts = np.clip(pts, [0,0], [IMG_SIZE[1]-1, IMG_SIZE[0]-1])
    pts = pts[np.argsort(pts[:,0])]
    
    if len(pts) < 6:
        return pts.astype(np.int32)
    
    try:
        tck, u = splprep([pts[:,0], pts[:,1]], s=curvature*20, k=1)  # s almost linear
        new_u = np.linspace(0, 1, 100)
        x_new, y_new = splev(new_u, tck)
        points = np.column_stack((np.array(x_new), np.array(y_new))).astype(np.int32)
        return points
    except:
        return pts.astype(np.int32)


def generate_piecewise_linear(n_segments=2):
    """Realistic segmented fault trace — long segments, tiny jogs"""
    pts = []
    x = random.uniform(80, IMG_SIZE[1]-80)
    y = random.uniform(80, IMG_SIZE[0]-80)
    direction = random.uniform(-np.pi, np.pi)
    
    for seg in range(n_segments):
        length = random.uniform(180, 450)  # long  segments
        jog_std = 0.08 if random.random() < 0.7 else 0.16  # mostly very straight
        direction += np.random.normal(0, jog_std)  # ~4.5–9° typical change
        dx = length * np.cos(direction)
        dy = length * np.sin(direction)
        x_end = x + dx
        y_end = y + dy
        pts.append([x, y])
        x = np.clip(x_end, 30, IMG_SIZE[1]-30)
        y = np.clip(y_end, 30, IMG_SIZE[0]-30)
    
    return np.array(pts, dtype=np.int32)


def generate_fault_network(n_lines_range=(4,12), high_curvature=False):
    n_lines = random.randint(*n_lines_range)
    lines = []
    
    for _ in range(n_lines):
        # Strong bias toward piecewise-linear / straight (real faults)
        if random.random() < 0.85 + (0.05 if not high_curvature else -0.10):
            n_seg = random.randint(1, 4) if not high_curvature else random.randint(2, 5)
            pts = generate_piecewise_linear(n_segments=n_seg)
        else:
            curv = random.uniform(0.0, 0.25) if not high_curvature else random.uniform(0.2, 0.6)
            pts = random_spline_points(n_points=random.randint(6,12), curvature=curv)
        
        if len(pts) < 30:
            continue
        
        if random.random() < 0.15:
            keep = random.uniform(0.80, 0.98)
            pts = pts[:int(len(pts)*keep)]
            if len(pts) < 30:
                continue
        
        lines.append(pts)
    
    straight_prob = 0.65 if not high_curvature else 0.45
    if random.random() < straight_prob:
        for _ in range(random.randint(2,6)):
            angle = random.uniform(0, np.pi*2)
            length = random.uniform(220, 500)
            cx = random.uniform(100, IMG_SIZE[1]-100)
            cy = random.uniform(100, IMG_SIZE[0]-100)
            x1 = int(cx - length/2 * np.cos(angle))
            y1 = int(cy - length/2 * np.sin(angle))
            x2 = int(cx + length/2 * np.cos(angle))
            y2 = int(cy + length/2 * np.sin(angle))
            straight = np.array([[x1,y1], [x2,y2]], dtype=np.int32)
            lines.append(straight)
    
    return lines


def rasterize_lines(canvas, lines, thickness_range=(2,7)):
    for pts in lines:
        thick = random.randint(*thickness_range)
        for i in range(len(pts)-1):
            t = random.randint(max(1, thick-2), thick+2)
            cv2.line(canvas, tuple(pts[i]), tuple(pts[i+1]), 255, t, lineType=cv2.LINE_AA)


def add_gaps(canvas, lines, gap_prob=0.35, max_gap_ratio=0.4):
    mask = np.zeros_like(canvas)
    rasterize_lines(mask, lines, thickness_range=(4,9))
    
    for pts in lines:
        total_len = len(pts)
        if total_len < 12:
            continue
        if random.random() > gap_prob:
            continue
        n_gaps = random.randint(1, min(3, total_len // 8))
        for _ in range(n_gaps):
            max_start = total_len - 10
            if max_start <= 0:
                continue
            start = random.randint(0, max_start)
            gap_len = int(random.uniform(0.08, max_gap_ratio) * total_len)
            gap_len = min(gap_len, total_len - start - 5)
            end = start + gap_len
            for i in range(start, end):
                if i < len(pts):
                    cv2.circle(mask, tuple(pts[i]), 12, 0, -1)
    
    canvas[mask == 0] = 0


def add_degradations(img, tier):
    if tier == "tier1_clean":
        return img
    
    noise = np.random.normal(0, random.uniform(8, 35), img.shape).astype(np.int16)
    img_noisy = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    if tier in ["tier2_noise_gaps", "tier3_curved_cross"]:
        blur_k = random.choice([1,3])
        if blur_k > 1:
            img_noisy = cv2.GaussianBlur(img_noisy, (blur_k,blur_k), 0)
        img_noisy = np.clip(img_noisy * random.uniform(0.5, 0.9), 0, 255).astype(np.uint8)
    
    elif tier == "tier4_dense_overlap":
        img_noisy = np.clip(img_noisy * random.uniform(0.4, 0.75), 0, 255).astype(np.uint8)
        stripe = np.sin(np.linspace(0, 60, IMG_SIZE[0]) * np.pi * 2) * 15
        img_noisy = np.clip(img_noisy + stripe[:,None].astype(np.int16), 0, 255).astype(np.uint8)
    
    elif tier == "tier5_hard":
        img_noisy = cv2.GaussianBlur(img_noisy, (3,5), 1.2)
        img_noisy = np.clip(img_noisy * random.uniform(0.3, 0.65), 0, 255).astype(np.uint8)
        salt_pepper = np.random.random(img.shape) < 0.015
        img_noisy[salt_pepper] = 255 if random.random() < 0.5 else 0
    
    return img_noisy


def main():
    for tier_idx, tier in enumerate(TIERS):
        print(f"Generating {NUM_IMAGES_PER_TIER} images for {tier} ...")
        tier_dir = os.path.join(OUTPUT_DIR, tier)
        
        for i in range(NUM_IMAGES_PER_TIER):
            canvas = np.zeros(IMG_SIZE, dtype=np.uint8)
            
            if tier == "tier1_clean":
                n_lines = (2,6); high_curv = False; gaps=False; thick=(4,8)
            elif tier == "tier2_noise_gaps":
                n_lines = (3,10); high_curv = False; gaps=True; thick=(3,7)
            elif tier == "tier3_curved_cross":
                n_lines = (5,14); high_curv = True; gaps=True; thick=(2,8)
            elif tier == "tier4_dense_overlap":
                n_lines = (12,28); high_curv = True; gaps=True; thick=(1,9)
            else:  # tier5_hard
                n_lines = (15,40); high_curv = True; gaps=True; thick=(1,10)
            
            lines = generate_fault_network(n_lines_range=n_lines, high_curvature=high_curv)
            
            rasterize_lines(canvas, lines, thickness_range=thick)
            
            if gaps:
                add_gaps(canvas, lines, gap_prob=0.4 if tier=="tier5_hard" else 0.25)
            
            final_img = add_degradations(canvas, tier)
            
            img_path = os.path.join(tier_dir, f"img_{i:04d}.png")
            cv2.imwrite(img_path, final_img)
            
            gt = [{"points": pts.tolist()} for pts in lines]
            gt_path = os.path.join(tier_dir, f"img_{i:04d}_gt.json")
            with open(gt_path, "w") as f:
                json.dump({"lines": gt, "image_size": list(IMG_SIZE)}, f, indent=2)
            
            if (i+1) % 20 == 0:
                print(f"  {i+1}/{NUM_IMAGES_PER_TIER}")
    
    print(f"\nDone! Images + GT saved in: {OUTPUT_DIR}")
    print("Each tier folder contains .png images and matching _gt.json vector ground truth.")

if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    main()

Generating 100 images for tier1_clean ...
  20/100
  40/100
  60/100
  80/100
  100/100
Generating 100 images for tier2_noise_gaps ...
  20/100
  40/100


c:\Users\dergun\miniforge3\envs\heatflow_work\Lib\site-packages\scipy\interpolate\_fitpack_py.py:159: RuntimeWarning: The maximal number of iterations (20) allowed for finding smoothing
spline with fp=s has been reached. Probable cause: s too small.
(abs(fp-s)/s>0.001)
  res = _impl.splprep(x, w, u, ub, ue, k, task, s, t, full_output, nest, per,


  60/100
  80/100
  100/100
Generating 100 images for tier3_curved_cross ...
  20/100
  40/100
  60/100
  80/100
  100/100
Generating 100 images for tier4_dense_overlap ...
  20/100
  40/100
  60/100
  80/100
  100/100
Generating 100 images for tier5_hard ...
  20/100
  40/100
  60/100
  80/100
  100/100

Done! Images + GT saved in: synthetic_fault_lines_realistic
Each tier folder contains .png images and matching _gt.json vector ground truth.
